# Process Behavior Demo: What does the system do and how it is done.

This notebook demonstrates the power of process behavior in enabling variation analysis (VAS).

**Key Concept: Time is for ordering only, not a factorial dimension.**

- `factors` define the cell structure (how data is grouped)
- `time` orders observations within each cell
- The user controls stratification explicitly via the `factors` list

## Examples Covered

| Factors | Time | Groups | Description |
|---------|------|--------|-------------|
| [lane] | pull | 4 | One chart per lane |
| [phase] | pull | 2 | One chart per phase |
| [lane, phase] | pull | 8 | One chart per lane/phase combination |

In [1]:
import pandas as pd
from processbehavior import ProcessBehavior

In [2]:
# Load fill weight manufacturing data
df = pd.read_csv('../processbehavior/datasets/data/FILLWEIGHTDATA_800.csv')

print(f"Dataset: {df.shape[0]} observations, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print()
print("Unique values:")
print(f"  lane: {df['lane'].unique().tolist()} ({df['lane'].nunique()} levels)")
print(f"  phase: {df['phase'].unique().tolist()} ({df['phase'].nunique()} levels)")
print(f"  pull: 1-{df['pull'].max()} ({df['pull'].nunique()} time points)")
print()
df.head(8)

Dataset: 800 observations, 4 columns
Columns: ['pull', 'lane', 'phase', 'fill_weight']

Unique values:
  lane: [1, 2, 3, 4] (4 levels)
  phase: [1, 2] (2 levels)
  pull: 1-100 (100 time points)



,pull,lane,phase,fill_weight
0,1,1,1,236.93
1,1,1,2,237.39
2,1,2,1,236.30
3,1,2,2,241.35
4,1,3,1,236.09
5,1,3,2,232.30
6,1,4,1,235.81
7,1,4,2,241.89


In [3]:
# Create ProcessBehavior - enables auto-complete
pb = ProcessBehavior(df)

---
## Example 1: Group by Lane

`factors=[lane]`, `time=pull`

Creates **4 charts** (one per lane), each with 200 observations over 100 time points.

In [4]:
# Formulate: factors=[lane], time=pull
study_lane = pb.formulate(
    response=pb.cols.fill_weight,
    factors=[pb.cols.lane],
    time=pb.cols.pull
)
print(study_lane)

╔══════════════════════════════════════════════════════════════════╗
║                        STUDY FORMULATION                         ║
╠══════════════════════════════════════════════════════════════════╣
║  Response: fill_weight                                           ║
║  Factors:  lane                                                  ║
║  Time:     pull                                                  ║
║  Precision: 3 decimal places                                     ║
╠══════════════════════════════════════════════════════════════════╣
║  Detected: SDS 1 - Full Factorial with Complete Replication      ║
║                                                                  ║
║  Valid Charts:  Xbar, S, R, Imr                                  ║
║  Recommended:   Xbar                                             ║
║  Residuals:     R2_S, R3_Xbar, R3_S, R4_Xbar, R4_S, R5_Xbar, R5_S║
╠══════════════════════════════════════════════════════════════════╣
║  Next: study.execute() or study.

In [5]:
# Execute with IMR (stratified individual charts)
result_lane = study_lane.execute(study_lane.charts.Imr)
print(result_lane)

ANALYSIS RESULT SUMMARY

Sampling Design State: SDS 1
Description: Full replication (all cells n≥2)

Analysis Type: Xbar
Response Variable: fill_weight
Grouping: lane
Time Variable: pull

Observations: 789
Charts: 1, 2, 3, 4
Stratified: Yes (4 groups)

Capabilities:
  Residuals: ✓
  Effects: ✓
  Interactions: ✓

⚠️  Signals: 5 points beyond limits


In [6]:
# Plot all 4 stratified charts
fig = result_lane.plot(
    show_zones=True,
    show_rules=True,
    ncols=2,
    width=1000,
    height=600,
    title='Stratified by Lane: 4 Charts (200 obs each)'
)
fig.show()

---
## Example 2: Group by Phase

`factors=[phase]`, `time=pull`

Creates **2 charts** (one per phase), each with 400 observations over 100 time points.

In [7]:
# Formulate: factors=[phase], time=pull
study_phase = pb.formulate(
    response=pb.cols.fill_weight,
    factors=[pb.cols.phase],
    time=pb.cols.pull
)
print(study_phase)

╔══════════════════════════════════════════════════════════════════╗
║                        STUDY FORMULATION                         ║
╠══════════════════════════════════════════════════════════════════╣
║  Response: fill_weight                                           ║
║  Factors:  phase                                                 ║
║  Time:     pull                                                  ║
║  Precision: 3 decimal places                                     ║
╠══════════════════════════════════════════════════════════════════╣
║  Detected: SDS 1 - Full Factorial with Complete Replication      ║
║                                                                  ║
║  Valid Charts:  Xbar, S, R, Imr                                  ║
║  Recommended:   Xbar                                             ║
║  Residuals:     R2_S, R3_Xbar, R3_S, R4_Xbar, R4_S, R5_Xbar, R5_S║
╠══════════════════════════════════════════════════════════════════╣
║  Next: study.execute() or study.

In [8]:
# Execute with IMR (stratified individual charts)
result_phase = study_phase.execute(study_phase.charts.Imr)
print(result_phase)

ANALYSIS RESULT SUMMARY

Sampling Design State: SDS 1
Description: Full replication (all cells n≥2)

Analysis Type: Xbar
Response Variable: fill_weight
Grouping: phase
Time Variable: pull

Observations: 789
Charts: 1, 2
Stratified: Yes (2 groups)

Capabilities:
  Residuals: ✓
  Effects: ✓
  Interactions: ✓

⚠️  Signals: 12 points beyond limits


In [9]:
# Plot 2 stratified charts side by side
fig = result_phase.plot(
    show_zones=True,
    show_rules=True,
    ncols=2,
    width=1000,
    height=400,
    title='Stratified by Phase: 2 Charts (400 obs each)'
)
fig.show()

---
## Example 3: Stratify by Lane and Phase

`factors=[lane, phase]`, `time=pull`

Creates **8 stratified charts** (one per lane/phase combination), each with 100 observations over 100 time points.

This is the most detailed view - each filling head gets its own control chart.

In [10]:
# Formulate: factors=[lane, phase], time=pull
study_lane_phase = pb.formulate(
    response=pb.cols.fill_weight,
    factors=[pb.cols.lane, pb.cols.phase],
    time=pb.cols.pull
)
print(study_lane_phase)

╔══════════════════════════════════════════════════════════════════╗
║                        STUDY FORMULATION                         ║
╠══════════════════════════════════════════════════════════════════╣
║  Response: fill_weight                                           ║
║  Factors:  lane, phase                                           ║
║  Time:     pull                                                  ║
║  Precision: 3 decimal places                                     ║
╠══════════════════════════════════════════════════════════════════╣
║  Detected: SDS 1 - Full Factorial with Complete Replication      ║
║                                                                  ║
║  Valid Charts:  Xbar, S, R, Imr                                  ║
║  Recommended:   Xbar                                             ║
║  Residuals:     R2_S, R3_Xbar, R3_S, R4_Xbar, R4_S, R5_Xbar, R5_S║
╠══════════════════════════════════════════════════════════════════╣
║  Next: study.execute() or study.

In [11]:
# Execute with IMR (stratified individual charts)
result_lane_phase = study_lane_phase.execute(study_lane_phase.charts.Imr)
print(result_lane_phase)

ANALYSIS RESULT SUMMARY

Sampling Design State: SDS 1
Description: Full replication (all cells n≥2)

Analysis Type: Xbar
Response Variable: fill_weight
Grouping: lane, phase
Time Variable: pull

Observations: 789
Charts: 1_1, 1_2, 2_1, 2_2, 3_1, 3_2, 4_1, 4_2
Stratified: Yes (8 groups)

Capabilities:
  Residuals: ✓
  Effects: ✓
  Interactions: ✓

⚠️  Signals: 66 points beyond limits


In [12]:
# Plot all 8 stratified charts
fig = result_lane_phase.plot(
    show_zones=True,
    show_rules=True,
    ncols=4,
    width=1400,
    height=700,
    title='Stratified by Lane x Phase: 8 Charts (100 obs each)'
)
fig.show()

In [13]:
# Detect signals across all 8 charts
signals = result_lane_phase.detect_signals()

print("Signal Detection Results")
print("=" * 40)
for chart_name, result in signals.items():
    if result.has_signals:
        n = len(result.violations)
        rules = result.violations.groupby('rule_name').size().to_dict()
        print(f"{chart_name}: {n} signals - {rules}")
    else:
        print(f"{chart_name}: No signals")

Signal Detection Results
1_1: 18 signals - {'rule_1': 6, 'rule_3': 3, 'rule_4': 7, 'rule_7': 2}
1_2: 10 signals - {'rule_1': 3, 'rule_2': 1, 'rule_3': 2, 'rule_4': 4}
2_1: 54 signals - {'rule_1': 10, 'rule_2': 5, 'rule_3': 21, 'rule_4': 12, 'rule_8': 6}
2_2: 74 signals - {'rule_1': 13, 'rule_2': 6, 'rule_3': 24, 'rule_4': 24, 'rule_8': 7}
3_1: 82 signals - {'rule_1': 6, 'rule_2': 5, 'rule_3': 34, 'rule_4': 26, 'rule_5': 2, 'rule_8': 9}
3_2: 59 signals - {'rule_1': 4, 'rule_2': 5, 'rule_3': 21, 'rule_4': 29}
4_1: 146 signals - {'rule_1': 11, 'rule_2': 10, 'rule_3': 48, 'rule_4': 66, 'rule_8': 11}
4_2: 162 signals - {'rule_1': 13, 'rule_2': 13, 'rule_3': 55, 'rule_4': 56, 'rule_5': 1, 'rule_8': 24}


---
## Summary: The Power of Stratification

| Formulation | Groups | Obs/Stratum | Use Case |
|------------|--------|-------------|----------|
| `factors=[lane]` | 4 | 200 | Compare lanes |
| `factors=[phase]` | 2 | 400 | Compare phases |
| `factors=[lane, phase]` | 8 | 100 | Monitor each head |

### Key Takeaways

1. **Time is for ordering only** - it sequences observations within each stratum
2. **Factors define stratification** - the user controls granularity
3. **Intuitive API** - just add variables to `factors` for finer stratification
4. **Automatic chart generation** - one specification creates all stratified charts

---
## Xbar/S Charts

Here each lane/phase combination becomes a subgroup.

In [14]:
# Xbar analysis: factors=[lane, phase], time=pull
# 8 subgroups (lane x phase) observed over 100 time points
result_xbar = study_lane_phase.execute(study_lane_phase.charts.Xbar)
print(result_xbar)

ANALYSIS RESULT SUMMARY

Sampling Design State: SDS 1
Description: Full replication (all cells n≥2)

Analysis Type: Xbar
Response Variable: fill_weight
Grouping: lane, phase
Time Variable: pull

Observations: 789
Charts: Xbar, Sbar

Capabilities:
  Residuals: ✓
  Effects: ✓
  Interactions: ✓

⚠️  Signals: 9 points beyond limits


In [15]:
# Plot Xbar chart (subgroup means)
fig = result_xbar.plot(
    chart='Xbar',
    show_zones=True,
    show_stats=True,
    show_rules=True,
    theme='ggplot',
    title='Xbar Chart: Mean Fill Weight by Lane/Phase'
)
fig.show()

# Display subgroup summary table with n values
print("\nSubgroup Summary (Xbar):")
result_xbar.chart_table('Xbar')


Subgroup Summary (Xbar):


,subgroup,n,value,center,lpl,upl,signal
0,1_1,99,238.111,237.785,237.395,238.174,
1,1_2,98,238.688,237.785,237.393,238.176,↑
2,2_1,100,237.395,237.785,237.397,238.172,↓
3,2_2,98,238.944,237.785,237.393,238.176,↑
4,3_1,99,236.498,237.785,237.395,238.174,↓
5,3_2,98,236.959,237.785,237.393,238.176,↓
6,4_1,99,237.587,237.785,237.395,238.174,
7,4_2,98,238.096,237.785,237.393,238.176,


In [16]:
# Plot S chart (subgroup standard deviations)
fig = result_xbar.plot(
    chart='Sbar',
    show_zones=True,
    show_stats=True,
    show_rules=True,
    theme='publication',
    title='S Chart: Variability by Lane/Phase'
)
fig.show()

# Display subgroup summary table - note n varies (98-100) due to missing values
print("\nSubgroup Summary (S Chart):")
print("Note: n varies across subgroups due to missing data removal")
result_xbar.chart_table('Sbar')


Subgroup Summary (S Chart):
Note: n varies across subgroups due to missing data removal


,subgroup,n,value,center,lpl,upl,signal
0,1_1,99,1.241,1.289,1.012,1.565,
1,1_2,98,0.977,1.289,1.011,1.567,↓
2,2_1,100,0.902,1.289,1.014,1.564,↓
3,2_2,98,1.043,1.289,1.011,1.567,
4,3_1,99,1.619,1.289,1.012,1.565,↑
5,3_2,98,1.637,1.289,1.011,1.567,↑
6,4_1,99,1.442,1.289,1.012,1.565,
7,4_2,98,1.449,1.289,1.011,1.567,


In [17]:
# R2_S chart - should match S chart exactly (same statistic, same limits)
print("Residual charts available:", study_lane_phase.residual_charts)

result_R2_S = study_lane_phase.execute(study_lane_phase.charts.R2_S)
fig = result_R2_S.plot(
    chart='R2_S',
    show_zones=True,
    show_stats=True,
    show_rules=True,
    theme='dark',
    title='S Chart using R2 residual: Variability by Lane/Phase'
)
fig.show()

# Display subgroup summary - compare with S chart above (should be identical)
print("\nSubgroup Summary (R2_S Chart):")
print("Compare with S chart above - values and limits should match exactly")
result_R2_S.chart_table('R2_S')

Residual charts available: ['R2_S', 'R3_Xbar', 'R3_S', 'R4_Xbar', 'R4_S', 'R5_Xbar', 'R5_S']



Subgroup Summary (R2_S Chart):
Compare with S chart above - values and limits should match exactly


,subgroup,n,value,center,lpl,upl,signal
0,1_1,99,1.241,1.289,1.012,1.565,
1,1_2,98,0.977,1.289,1.011,1.567,↓
2,2_1,100,0.902,1.289,1.014,1.564,↓
3,2_2,98,1.043,1.289,1.011,1.567,
4,3_1,99,1.619,1.289,1.012,1.565,↑
5,3_2,98,1.637,1.289,1.011,1.567,↑
6,4_1,99,1.442,1.289,1.012,1.565,
7,4_2,98,1.449,1.289,1.011,1.567,


---
## Effects and Interactions Analysis

The Xbar/S analysis also calculates main effects and interactions from the VAS (Variance Analysis System) framework.

**Main Effects** show how each factor level differs from the grand mean:
- Positive = above average
- Negative = below average

**Interaction Effects** show whether factor combinations behave differently than expected from main effects alone.

In [18]:
# Main Effects by Factor
print("=" * 50)
print("MAIN EFFECTS BY LANE")
print("=" * 50)
print("How each lane differs from the grand mean:")
print(result_xbar.effects['lane'].to_string(index=False))

print("\n" + "=" * 50)
print("MAIN EFFECTS BY PHASE")
print("=" * 50)
print("How each phase differs from the grand mean:")
print(result_xbar.effects['phase'].to_string(index=False))

MAIN EFFECTS BY LANE
How each lane differs from the grand mean:
 lane  Main_Effect
    1     0.615803
    2     0.379297
    3    -1.055162
    4     0.058138

MAIN EFFECTS BY PHASE
How each phase differs from the grand mean:
 phase  Main_Effect
     1    -0.384562
     2     0.389467


In [19]:
# Interaction Effects (Lane × Phase)
print("=" * 50)
print("INTERACTION EFFECTS (Lane × Phase)")
print("=" * 50)
print("Rx > 0: combination performs better than expected from main effects")
print("Rx < 0: combination performs worse than expected from main effects")
print()
print(result_xbar.effects['F1xF2'].to_string(index=False))

INTERACTION EFFECTS (Lane × Phase)
Rx > 0: combination performs better than expected from main effects
Rx < 0: combination performs worse than expected from main effects

 lane  phase        Rx
    1      1  0.097803
    1      2 -0.099782
    2      1 -0.382405
    2      2  0.393152
    3      1  0.155333
    3      2 -0.157899
    4      1  0.131226
    4      2 -0.133546


In [20]:
# Combined Main Effect by Subgroup
print("=" * 50)
print("COMBINED MAIN EFFECT BY SUBGROUP (Lane_Phase)")
print("=" * 50)
print("Total deviation from grand mean for each lane/phase combination:")
print()
print(result_xbar.effects['main_effect'].to_string(index=False))

COMBINED MAIN EFFECT BY SUBGROUP (Lane_Phase)
Total deviation from grand mean for each lane/phase combination:

rsg  Main_Effect
1_1     0.329044
1_2     0.905487
2_1    -0.387670
2_2     1.161916
3_1    -1.284390
3_2    -0.823595
4_1    -0.195198
4_2     0.314058


In [21]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create main effects bar charts
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Main Effect by Lane', 'Main Effect by Phase'),
    horizontal_spacing=0.15
)

# Lane effects
lane_effects = result_xbar.effects['lane']
colors = ['#2ecc71' if x >= 0 else '#e74c3c' for x in lane_effects['Main_Effect']]
fig.add_trace(
    go.Bar(
        x=lane_effects['lane'].astype(str),
        y=lane_effects['Main_Effect'],
        marker_color=colors,
        name='Lane',
        showlegend=False
    ),
    row=1, col=1
)

# Phase effects
phase_effects = result_xbar.effects['phase']
colors = ['#2ecc71' if x >= 0 else '#e74c3c' for x in phase_effects['Main_Effect']]
fig.add_trace(
    go.Bar(
        x=phase_effects['phase'].astype(str),
        y=phase_effects['Main_Effect'],
        marker_color=colors,
        name='Phase',
        showlegend=False
    ),
    row=1, col=2
)

fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=2)

fig.update_layout(
    title='Main Effects: Deviation from Grand Mean',
    height=400,
    width=800
)
fig.update_xaxes(title_text="Lane", row=1, col=1)
fig.update_xaxes(title_text="Phase", row=1, col=2)
fig.update_yaxes(title_text="Main Effect", row=1, col=1)
fig.update_yaxes(title_text="Main Effect", row=1, col=2)

fig.show()

In [22]:
# Interaction Plot: How lane effect varies by phase
# Parallel lines = no interaction; crossing lines = interaction

interaction_df = result_xbar.effects['F1xF2'].copy()

fig = go.Figure()

# Plot each phase as a line across lanes
for phase in [1, 2]:
    phase_data = interaction_df[interaction_df['phase'] == phase]
    fig.add_trace(go.Scatter(
        x=phase_data['lane'].astype(str),
        y=phase_data['Rx'],
        mode='lines+markers',
        name=f'Phase {phase}',
        line=dict(width=3),
        marker=dict(size=10)
    ))

fig.add_hline(y=0, line_dash="dash", line_color="gray")

fig.update_layout(
    title='Interaction Plot: Lane × Phase',
    xaxis_title='Lane',
    yaxis_title='Interaction Effect (Rx)',
    height=400,
    width=700,
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99)
)

fig.show()

print("\nInterpretation:")
print("- Lines that cross or diverge indicate interaction between factors")
print("- Parallel lines would indicate no interaction (additive effects only)")
print("- Here we see Lane 2 behaves differently: Phase 2 performs much better than Phase 1")


Interpretation:
- Lines that cross or diverge indicate interaction between factors
- Parallel lines would indicate no interaction (additive effects only)
- Here we see Lane 2 behaves differently: Phase 2 performs much better than Phase 1


In [23]:
# Combined Main Effects by Subgroup
main_effect_df = result_xbar.effects['main_effect'].copy()
colors = ['#2ecc71' if x >= 0 else '#e74c3c' for x in main_effect_df['Main_Effect']]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=main_effect_df['rsg'],
    y=main_effect_df['Main_Effect'],
    marker_color=colors,
    text=main_effect_df['Main_Effect'].round(2),
    textposition='outside'
))

fig.add_hline(y=0, line_dash="dash", line_color="gray")

fig.update_layout(
    title='Combined Main Effect by Subgroup (Lane_Phase)',
    xaxis_title='Subgroup (Lane_Phase)',
    yaxis_title='Main Effect',
    height=400,
    width=900
)

fig.show()

print("\nKey Insights:")
print("- Subgroups 3_1 and 3_2 (Lane 3) are consistently below average")
print("- Subgroup 2_2 (Lane 2, Phase 2) shows the highest positive effect")
print("- Lane 3 appears to be the main driver of variation in this process")


Key Insights:
- Subgroups 3_1 and 3_2 (Lane 3) are consistently below average
- Subgroup 2_2 (Lane 2, Phase 2) shows the highest positive effect
- Lane 3 appears to be the main driver of variation in this process
